# Imports

In [0]:
# init

from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, window

# Read tables from silver_schema

In [0]:
df_crm_cust_info = spark.table("workspace.silver_schema.crm_cust_info")
display(df_crm_cust_info)
df_crm_cust_info.createOrReplaceTempView("customer_info")

In [0]:
df_erp_cust_az12 = spark.table("workspace.silver_schema.erp_cust_az12")
display(df_erp_cust_az12)
df_erp_cust_az12.createOrReplaceTempView("customer_gender_birthdate")

In [0]:
df_erp_loc_a101 = spark.table("workspace.silver_schema.erp_loc_a101")
display(df_erp_loc_a101)
df_erp_loc_a101.createOrReplaceTempView("location")

# Apply Business Rule for Gold Layer

In [0]:
query = """
    
    SELECT 
    /*
        ci.customer_id,
        ci.customer_key,
        ci.customer_firstname,
        ci.customer_lastname,
        ci.customer_marital_status,
        ci.customer_gender,
        ci.customer_create_date,
        cgb.customer_birthdate,
        cgb.customer_gender
    */
        ci.*,
        cgb.*,
        loc.*
    FROM customer_info ci
    LEFT JOIN customer_gender_birthdate cgb
        ON ci.customer_key = cgb.customer_key
    
    LEFT JOIN location loc
        ON ci.customer_key = loc.customer_key
"""

spark.sql(query).display()

In [0]:
"""
    We have two gender columns. Apply data enrichement using both columns.
"""

query = """
    
    SELECT 
    DISTINCT
    ci.customer_gender,
    cgb.customer_gender
    FROM customer_info ci
    LEFT JOIN customer_gender_birthdate cgb
    ON ci.customer_key = cgb.customer_key
    -- WHERE ci.customer_gender <> cgb.customer_gender
    WHERE ci.customer_gender IS DISTINCT FROM cgb.customer_gender
    ORDER BY 1,2
"""

spark.sql(query).display()

"""
    Notes and Observations
    WHERE ci.customer_gender IS DISTINCT FROM cgb.customer_gender and WHERE ci.customer_gender <> cgb.customer_gender differ in result.

"""

In [0]:
query = """

    SELECT * FROM
    (
        SELECT 
            ROW_NUMBER() OVER(ORDER BY ci.customer_id) AS primary_key,
            ci.customer_id,
            ci.customer_key,
            ci.customer_firstname AS first_name,
            ci.customer_lastname AS last_name,
            CASE  
            /*
                New gender column is an integration of both gender columns from the ci and the cgb tables
            */
                WHEN ci.customer_gender = 'n/a' THEN COALESCE(cgb.customer_gender, 'n/a')
                ELSE ci.customer_gender
            END AS gender,
            cgb.customer_birthdate AS birth_date,
            ci.customer_marital_status AS marital_status,
            loc.customer_country AS country,
            ci.customer_create_date AS created_date
            
        FROM customer_info ci
        LEFT JOIN customer_gender_birthdate cgb
            ON ci.customer_key = cgb.customer_key

        LEFT JOIN location as loc
            ON ci.customer_key = loc.customer_key
    )


"""
df_dim_customer = spark.sql(query)

df_dim_customer.display()

# Write to gold_schema

In [0]:
df_dim_customer.write.mode('overwrite').format('delta').saveAsTable("workspace.gold_schema.dim_customer")

# Read dim_customer from gold layer

In [0]:
df_gold_dim_customer = spark.table("workspace.gold_schema.dim_customer")
df_gold_dim_customer.display()